In [1]:
from typing import List, Tuple
import json
import re
import math
import os

import torch
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from generate import extract_first_integer

from datageneration.util import compute_all_op_pert_erroneous_answers
from stat_util import wilson_score_interval, two_proportion_z_test, pointwise_mutual_info

/home/yanick/miniforge3/envs/csm-mwps/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Comparison Inconsistency

## Evaluation

In [2]:
category = "ci"
series = "ls500"
oppert_file = f"data/datasets/{category}/icl/{series}/distractors_oppert.json"
res_file_path = f"data/datasets/{category}/icl/all_results_{series}.csv"
res_file_path_3shot = f"data/datasets/{category}/icl/all_results_{series}_3s.csv"

models = ["llama31_8B_I", "llama2_7B_I", "llama31_70B_I", "qwen25_7B_I"]
shots = ["s3"]
varys = ["mwp", "instva"]
rtfes = [True, False]
rts = [True, False]
cots = [True, False]
hles = [True, False]

eval_configs = [
    {
        "model": model,
        "shots": shot,
        "vary": vary,
        "rtfe": rtfe,
        "rt": rt,
        "cot": cot,
        "hle": hle
    }
    for model in models
    for shot in shots
    for vary in varys
    for rtfe in rtfes
    for rt in rts
    for cot in cots
    for hle in hles
]

In [3]:
category = "ci"
series = "ls100"
oppert_file = f"data/datasets/{category}/icl/{series}/distractors_oppert.json"
res_file_path = f"data/datasets/{category}/icl/all_results_{series}.csv"
res_file_path_3shot = f"data/datasets/{category}/icl/all_results_{series}_3s.csv"

models = ["llama31_8B_I", "llama2_7B_I", "llama31_70B_I", "qwen25_7B_I"]
shots = ["s0-5s10"]
varys = ["mwp", "instva"]
rtfes = [True, False]
rts = [True, False]
cots = [True, False]
hles = [True, False]

eval_configs = [
    {
        "model": model,
        "shots": shot,
        "vary": vary,
        "rtfe": rtfe,
        "rt": rt,
        "cot": cot,
        "hle": hle
    }
    for model in models
    for shot in shots
    for vary in varys
    for rtfe in rtfes
    for rt in rts
    for cot in cots
    for hle in hles
]

In [4]:
category = "am"
series = "500"
oppert_file = f"data/datasets/{category}/icl/{series}/distractors_oppert.json"
res_file_path = f"data/datasets/{category}/icl/all_results_{series}.csv"
res_file_path_3shot = f"data/datasets/{category}/icl/all_results_{series}_3s.csv"

models = ["llama31_8B_I", "llama2_7B_I", "llama31_70B_I", "qwen25_7B_I"]
shots = ["s3"]
varys = ["mwp", "instva"]
rtfes = [True, False]
rts = [True, False]
cots = [True, False]
hles = [True, False]

eval_configs = [
    {
        "model": model,
        "shots": shot,
        "vary": vary,
        "rtfe": rtfe,
        "rt": rt,
        "cot": cot,
        "hle": hle
    }
    for model in models
    for shot in shots
    for vary in varys
    for rtfe in rtfes
    for rt in rts
    for cot in cots
    for hle in hles
]

In [5]:
category = "am"
series = "100"
oppert_file = f"data/datasets/{category}/icl/{series}/distractors_oppert.json"
res_file_path = f"data/datasets/{category}/icl/all_results_{series}.csv"
res_file_path_3shot = f"data/datasets/{category}/icl/all_results_{series}_3s.csv"

models = ["llama31_8B_I", "llama2_7B_I", "llama31_70B_I", "qwen25_7B_I"]
shots = ["s0s1s2s3s4s5", "s0s1s2s3s4s5s10"]
varys = ["mwp", "instva"]
rtfes = [True, False]
rts = [True, False]
cots = [True, False]
hles = [True, False]

eval_configs = [
    {
        "model": model,
        "shots": shot,
        "vary": vary,
        "rtfe": rtfe,
        "rt": rt,
        "cot": cot,
        "hle": hle
    }
    for model in models
    for shot in shots
    for vary in varys
    for rtfe in rtfes
    for rt in rts
    for cot in cots
    for hle in hles
]

In [6]:
empty_response_for_mwp_3shot = set([])

In [7]:
oppert_cache = {}
if oppert_file is not None:
    with open(oppert_file, "r+") as f:
        oppert_data = json.load(f)
    
    for mwp_id in oppert_data:
        mwp_data = oppert_data[mwp_id]
        for inst_id in mwp_data["instantiations"]:
            inst_data = mwp_data["instantiations"][inst_id]
            oppert_cache[inst_data["problem"]] = inst_data["all_op_perturbation_answers"]

In [8]:
def apply_taxonomy(answer, inst_data) -> Tuple[bool, bool, bool, bool, bool]:
    correct_answer = inst_data["correct_answer"]["answer"]
    misconception_answers = inst_data["misconception_answers"]

    match_plausible_any = any([
        d["answer"] == answer
        for d in misconception_answers
        if d["plausible"] and d["answer"] != correct_answer
    ])

    match_plausible_all = (answer == sorted(misconception_answers, key=(lambda x: len(x["misconceptions"])), reverse=True)[0]["answer"])
    match_oppert = (answer in inst_data["all_op_perturbation_answers"] and answer != correct_answer)
    match_given = (answer in inst_data["instantiation"].values() and not match_plausible_all) # TODO: change this to match_plausible_all if we use that def
    match_correct = (answer == correct_answer)

    return match_plausible_any, match_plausible_all, match_oppert, match_given, match_correct

def load_and_eval(series, config, ignore_mwps: List[int], out_file: str):
    model = config["model"]
    shots = config["shots"]
    vary = config["vary"]
    rtfe = config["rtfe"]
    rt = config["rt"]
    cot = config["cot"]
    hle = config["hle"]

    # Load data
    filename = f"{model}_{shots}_{vary}"
    if not rt:
        filename += "_nort"
    if not cot:
        filename += "_nocot"
    if rtfe:
        filename += "_rtfe"
    if hle:
        filename += "_hle"

    full_file = f"data/datasets/{category}/icl/{series}/{filename}.json"
    with open(full_file, "r+") as f:
        data = json.load(f)
        print(f"Loaded dataset of {len(data)} MWPs from {full_file}")

    def identical_structure(mwp_id_1, mwp_id_2):
        return data[mwp_id_1]["metadata"]["correct_expression"] == data[mwp_id_2]["metadata"]["correct_expression"]
    
    def identical_pattern(mwp_id_1, inst_id_1, mwp_id_2, inst_id_2):
        expressions_1 = set([ma.get("expression", None) for ma in data[mwp_id_1]["instantiations"][inst_id_1]["misconception_answers"]])
        expressions_2 = set([ma.get("expression", None) for ma in data[mwp_id_2]["instantiations"][inst_id_2]["misconception_answers"]])
        return expressions_1 == expressions_2

    # Analyze data
    nr_sol_ans_not_empty_by_shots = {}
    nr_dist_ans_not_empty_by_shots = {}
    nr_by_shots = {}

    nr_sol_correct_by_shots = {}
    nr_sol_plausible_any_by_shots = {}
    nr_sol_plausible_all_by_shots = {}
    nr_sol_oppert_by_shots = {}
    nr_sol_given_by_shots = {}

    nr_distr_plausible_any_by_shots = {} # makes error on any subset of applicable rules
    nr_distr_plausible_all_by_shots = {} # makes error on all applicable rules
    nr_distr_oppert_by_shots = {} # any flip from + to - or vice versa
    nr_distr_given_by_shots = {} # given number but not equal to a plausible distractor
    nr_distr_correct_by_shots = {} # correct solution
    nr_distr_plausible_any_not_id_struct_by_shots = {}
    nr_distr_plausible_not_any_id_pattern_by_shots = {}
    nr_distr_plausible_any_not_all_id_pattern_by_shots = {}

    nr_sol_correct_and_distr_plausible_any_by_shots =  {}

    perplexities_distractor_reasoning_by_shots = {}
    perplexities_distractor_answer_by_shots = {}
    perplexities_answer_reasoning_by_shots = {}
    perplexities_answer_answer_by_shots = {}

    solved_by_shots = {}
    plausible_any_by_shots = {}
    plausible_all_by_shots = {}
    not_id_struct_by_shots = {}
    not_any_id_pattern_by_shots = {}
    not_all_id_pattern_by_shots = {}

    for mwp_id in data:
        if mwp_id in ignore_mwps: continue

        mwp_data = data[mwp_id]

        for inst_id in mwp_data["instantiations"]:
            inst_data = mwp_data["instantiations"][inst_id]
            if not ("llm_answer" in inst_data and "llm_distractor" in inst_data): continue
            if inst_data.get("all_op_perturbation_answers", None) is None:
                if inst_data["problem"] not in oppert_cache:
                    inst_data["all_op_perturbation_answers"] = compute_all_op_pert_erroneous_answers(mwp_data["metadata"]["correct_expression"], inst_data["instantiation"])
                    oppert_cache[inst_data["problem"]] = inst_data["all_op_perturbation_answers"]
                else:
                    inst_data["all_op_perturbation_answers"] = oppert_cache[inst_data["problem"]]
            
            correct_answer = inst_data["correct_answer"]["answer"]
            for shots in inst_data["llm_answer"]:
                nr_by_shots[shots] = nr_by_shots.get(shots, 0) + 1
                
                llm_answer = extract_first_integer(inst_data["llm_answer"][shots]["answer"])
                llm_distractor = extract_first_integer(inst_data["llm_distractor"][shots]["answer"])

                sol_non_empty = False
                if inst_data["llm_answer"][shots]["answer"]:
                    nr_sol_ans_not_empty_by_shots[shots] = nr_sol_ans_not_empty_by_shots.get(shots, 0) + 1
                    sol_non_empty = True

                dist_non_empty = False
                if inst_data["llm_distractor"][shots]["answer"]:
                    nr_dist_ans_not_empty_by_shots[shots] = nr_dist_ans_not_empty_by_shots.get(shots, 0) + 1
                    dist_non_empty = True

                if not dist_non_empty and int(shots) == 3:
                    empty_response_for_mwp_3shot.add(mwp_id)

                # solved
                solved = llm_answer == correct_answer
                if solved:
                    nr_sol_correct_by_shots[shots] = nr_sol_correct_by_shots.get(shots, 0) + 1
                else:
                    # suggested solution matches any plausible distractor
                    sol_match_plausible_any, sol_match_plausible_all, sol_match_oppert, sol_match_given, sol_match_correct = apply_taxonomy(llm_answer, inst_data)

                    nr_sol_plausible_any_by_shots[shots] = nr_sol_plausible_any_by_shots.get(shots, 0) + sol_match_plausible_any
                    nr_sol_plausible_all_by_shots[shots] = nr_sol_plausible_all_by_shots.get(shots, 0) + sol_match_plausible_all
                    nr_sol_oppert_by_shots[shots] = nr_sol_oppert_by_shots.get(shots, 0) + sol_match_oppert
                    nr_sol_given_by_shots[shots] = nr_sol_given_by_shots.get(shots, 0) + sol_match_given


                # suggested distractor matches inconsistent one
                match_plausible_any, match_plausible_all, match_oppert, match_given, match_correct = apply_taxonomy(llm_distractor, inst_data)
                
                nr_distr_plausible_all_by_shots[shots] = nr_distr_plausible_all_by_shots.get(shots, 0) + match_plausible_all
                nr_distr_plausible_any_by_shots[shots] = nr_distr_plausible_any_by_shots.get(shots, 0) + match_plausible_any
                nr_distr_oppert_by_shots[shots] = nr_distr_oppert_by_shots.get(shots, 0) + match_oppert
                nr_distr_given_by_shots[shots] = nr_distr_given_by_shots.get(shots, 0) + match_given
                nr_distr_correct_by_shots[shots] = nr_distr_correct_by_shots.get(shots, 0) + match_correct
                
                nr_sol_correct_and_distr_plausible_any_by_shots[shots] = nr_sol_correct_and_distr_plausible_any_by_shots.get(shots, 0) + (match_plausible_any and solved)

                not_identical_structure = not any([identical_structure(mwp_id, ex_mwp_id) for ex_mwp_id,ex_inst_id in inst_data["llm_distractor"][shots].get("example_ids", [])])
                if not_identical_structure:
                    # none of the in-context examples has the same problem structure
                    if match_plausible_any:
                        nr_distr_plausible_any_not_id_struct_by_shots[shots] = nr_distr_plausible_any_not_id_struct_by_shots.get(shots, 0) + 1
                    not_id_struct_by_shots[shots] = not_id_struct_by_shots.get(shots, 0) + 1

                if not any([identical_pattern(mwp_id, inst_id, ex_mwp_id, ex_inst_id) for ex_mwp_id,ex_inst_id in inst_data["llm_distractor"][shots].get("example_ids", [])]):
                    # none of the in-context examples has the same structure and error pattern
                    if match_plausible_any:
                        nr_distr_plausible_not_any_id_pattern_by_shots[shots] = nr_distr_plausible_not_any_id_pattern_by_shots.get(shots, 0) + 1
                    not_any_id_pattern_by_shots[shots] = not_any_id_pattern_by_shots.get(shots, 0) + 1

                if not all([identical_pattern(mwp_id, inst_id, ex_mwp_id, ex_inst_id) for ex_mwp_id,ex_inst_id in inst_data["llm_distractor"][shots].get("example_ids", [])]):
                    # none of the in-context examples has the same structure and error pattern
                    if match_plausible_any:
                        nr_distr_plausible_any_not_all_id_pattern_by_shots[shots] = nr_distr_plausible_any_not_all_id_pattern_by_shots.get(shots, 0) + 1
                    not_all_id_pattern_by_shots[shots] = not_all_id_pattern_by_shots.get(shots, 0) + 1

                # avg perplexity on solution
                if inst_data["llm_answer"][shots].get("reasoning_perplexity", None) is not None:
                    perplexities_answer_reasoning_by_shots[shots] = perplexities_answer_reasoning_by_shots.get(shots, []) + [inst_data["llm_answer"][shots]["reasoning_perplexity"]]
                    perplexities_answer_answer_by_shots[shots] = perplexities_answer_answer_by_shots.get(shots, []) + [inst_data["llm_answer"][shots]["answer_perplexity"]]

                # avg perplexity on distractors
                if inst_data["llm_distractor"][shots].get("reasoning_perplexity", None) is not None:
                    # if llm_distractor == correct_answer:
                    perplexities_distractor_reasoning_by_shots[shots] = perplexities_distractor_reasoning_by_shots.get(shots, []) + [inst_data["llm_distractor"][shots]["reasoning_perplexity"]]
                    perplexities_distractor_answer_by_shots[shots] = perplexities_distractor_answer_by_shots.get(shots, []) + [inst_data["llm_distractor"][shots]["answer_perplexity"]]
                
                if sol_non_empty and dist_non_empty:
                    solved_by_shots[shots] = solved_by_shots.get(shots, []) + [solved]
                    plausible_any_by_shots[shots] = plausible_any_by_shots.get(shots, []) + [match_plausible_any]
                    plausible_all_by_shots[shots] = plausible_all_by_shots.get(shots, []) + [match_plausible_all]

    # store CSV
    if os.path.exists(out_file):
        all_results_series = pd.read_csv(out_file)
    else:
        all_results_series = pd.DataFrame(columns=[
            "llm",
            "shots",
            "rt",
            "vary",
            "cot", 
            "rtfe",
            "hle",
            "sol_correct",
            "sol_plausible_any",
            "sol_plausible_all",
            "sol_oppert",
            "sol_given",
            "distr_plausible_any",
            "distr_plausible_all",
            "distr_oppert",
            "distr_correct",
            "distr_given",
            "distr_plausible_not_id_struct",
            "distr_plausible_not_any_id_pattern",
            "distr_plausible_not_all_id_pattern",
            "sol_correct_n_distr_plausible_any",
            "pmi_solved_plausible",
            "n",
            "n_nonempty_solve",
            "n_nonempty_distgen",
            "n_not_id_struct",
            "n_not_any_id_pattern",
            "n_not_all_id_pattern"
        ])

    # store data
    for shots in nr_by_shots.keys():
        new_row = pd.DataFrame([{
            "llm": model,
            "shots": shots,
            "rt": rt,
            "vary": vary,
            "cot": cot,
            "rtfe": rtfe,
            "hle": hle,
            "sol_correct": nr_sol_correct_by_shots.get(shots, 0),
            "sol_plausible_any": nr_sol_plausible_any_by_shots.get(shots, 0),
            "sol_plausible_all": nr_sol_plausible_all_by_shots.get(shots, 0),
            "sol_oppert": nr_sol_oppert_by_shots.get(shots, 0),
            "sol_given": nr_sol_given_by_shots.get(shots, 0),
            "distr_plausible_any": nr_distr_plausible_any_by_shots.get(shots, 0),
            "distr_plausible_all": nr_distr_plausible_all_by_shots.get(shots, 0),
            "distr_oppert": nr_distr_oppert_by_shots.get(shots, 0),
            "distr_correct": nr_distr_correct_by_shots.get(shots, 0),
            "distr_given": nr_distr_given_by_shots.get(shots, 0),
            "distr_plausible_not_id_struct": nr_distr_plausible_any_not_id_struct_by_shots.get(shots, 0),
            "distr_plausible_not_any_id_pattern": nr_distr_plausible_not_any_id_pattern_by_shots.get(shots, 0),
            "distr_plausible_not_all_id_pattern": nr_distr_plausible_any_not_all_id_pattern_by_shots.get(shots, 0),
            "sol_correct_n_distr_plausible_any": nr_sol_correct_and_distr_plausible_any_by_shots.get(shots, 0),
            "pmi_solved_plausible": pointwise_mutual_info(solved_by_shots.get(shots, []), plausible_any_by_shots.get(shots, [])),
            "n": nr_by_shots.get(shots, 0),
            "n_nonempty_solve": nr_sol_ans_not_empty_by_shots.get(shots, 0),
            "n_nonempty_distgen": nr_dist_ans_not_empty_by_shots.get(shots, 0),
            "n_not_id_struct": not_id_struct_by_shots.get(shots, 0),
            "n_not_any_id_pattern": not_any_id_pattern_by_shots.get(shots, 0),
            "n_not_all_id_pattern": not_all_id_pattern_by_shots.get(shots, 0)
        }])

        all_results_series = pd.concat([all_results_series, new_row], ignore_index=True)

    all_results_series.to_csv(out_file, index=False)

In [9]:
empty_response_for_mwp_3shot.clear()

In [10]:
import traceback
if os.path.exists(res_file_path):
    os.remove(res_file_path)
for conf in eval_configs:
    try:
        load_and_eval(series, conf, {}, res_file_path)
    except Exception as e:
        if not "Errno 2" in str(e):
            print(e.with_traceback())
        continue

Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_8B_I_s0s1s2s3s4s5s10_mwp_rtfe.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_8B_I_s0s1s2s3s4s5s10_mwp_hle.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_8B_I_s0s1s2s3s4s5s10_mwp.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_8B_I_s0s1s2s3s4s5s10_mwp_nort_hle.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_8B_I_s0s1s2s3s4s5s10_mwp_nort.json


/tmp/ipykernel_19914/2932580644.py:248: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_results_series = pd.concat([all_results_series, new_row], ignore_index=True)


Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama2_7B_I_s0s1s2s3s4s5_mwp_rtfe.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama2_7B_I_s0s1s2s3s4s5_mwp_hle.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama2_7B_I_s0s1s2s3s4s5_mwp.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama2_7B_I_s0s1s2s3s4s5_mwp_nort_hle.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama2_7B_I_s0s1s2s3s4s5_mwp_nort.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_70B_I_s0s1s2s3s4s5s10_mwp_rtfe.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_70B_I_s0s1s2s3s4s5s10_mwp_hle.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_70B_I_s0s1s2s3s4s5s10_mwp.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_70B_I_s0s1s2s3s4s5s10_mwp_nort_hle.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_70B_I_s0s1s2s3s4s5s10_mwp_nort.json
Loaded dataset of 1

In [11]:
# NOTE: make sure empty_response_for_mwp_5shot is already populated correctly!
if os.path.exists(res_file_path_3shot):
    os.remove(res_file_path_3shot)
for conf in eval_configs:
    try:
        load_and_eval(series, conf, empty_response_for_mwp_3shot, res_file_path_3shot)
    except Exception as e:
        if not "Errno 2" in str(e):
            print(e.with_traceback())
        continue

Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_8B_I_s0s1s2s3s4s5s10_mwp_rtfe.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_8B_I_s0s1s2s3s4s5s10_mwp_hle.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_8B_I_s0s1s2s3s4s5s10_mwp.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_8B_I_s0s1s2s3s4s5s10_mwp_nort_hle.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_8B_I_s0s1s2s3s4s5s10_mwp_nort.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama2_7B_I_s0s1s2s3s4s5_mwp_rtfe.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama2_7B_I_s0s1s2s3s4s5_mwp_hle.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama2_7B_I_s0s1s2s3s4s5_mwp.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama2_7B_I_s0s1s2s3s4s5_mwp_nort_hle.json


/tmp/ipykernel_19914/2932580644.py:248: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_results_series = pd.concat([all_results_series, new_row], ignore_index=True)


Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama2_7B_I_s0s1s2s3s4s5_mwp_nort.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_70B_I_s0s1s2s3s4s5s10_mwp_rtfe.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_70B_I_s0s1s2s3s4s5s10_mwp_hle.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_70B_I_s0s1s2s3s4s5s10_mwp.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_70B_I_s0s1s2s3s4s5s10_mwp_nort_hle.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/llama31_70B_I_s0s1s2s3s4s5s10_mwp_nort.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/qwen25_7B_I_s0s1s2s3s4s5s10_mwp_rtfe.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/qwen25_7B_I_s0s1s2s3s4s5s10_mwp_hle.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/qwen25_7B_I_s0s1s2s3s4s5s10_mwp.json
Loaded dataset of 100 MWPs from data/datasets/am/icl/100/qwen25_7B_I_s0s1s2s3s4s5s10_mwp_nort_hle.json
Loaded 

In [12]:
len(empty_response_for_mwp_3shot)

0